In [2]:
import polars as pl
import duckdb
import matplotlib.pyplot as plt

In [5]:
query_yellow_2009 = """
WITH CTE_yellow_2009 AS (
    SELECT 
        Trip_Pickup_DateTime AS pick_up_time,
        Trip_Dropoff_DateTime AS drop_off_time,
        Passenger_Count AS passenger_count,
        Trip_Distance AS trip_distance,
        Payment_Type AS payment_type,
        Total_Amt AS total_amount,
        Tip_Amt AS tip_amount,
        CASE Payment_Type
            WHEN  'Credit' THEN 1
            WHEN  'CREDIT' THEN 1
            ELSE 2
        END AS payment_category
    FROM 'C:/Users/ekadw/Documents/DATA/NY_Taxi/2009/yellow_taxi_2009/yellow_tripdata_*.parquet'
    WHERE Trip_Pickup_DateTime IS NOT NULL
        AND Trip_Dropoff_DateTime IS NOT NULL
        AND (Passenger_Count >= 0)
        AND (Trip_Distance >= 0)
        AND (Trip_Distance <= 50)
        AND Payment_Type IS NOT NULL
        AND (Total_Amt >= 0)
        AND (Tip_Amt IS >= 0)
        AND (Trip_Pickup_DateTime >= '2009-01-01') 
        AND (Trip_Pickup_DateTime < '2010-10-01')
)

SELECT 
    passenger_count,
    trip_distance,
    total_amount
FROM CTE_yellow_2009
"""

con = duckdb.connect()
df_yellow_2009 = con.execute(query_yellow_2009).fetchdf()
df_yellow_2009.head()

ParserException: Parser Error: syntax error at or near ">="

In [ ]:
query_classification = """
WITH CTE_yellow_2009 AS (
    SELECT 
        Trip_Pickup_DateTime AS pick_up_time,
        Trip_Dropoff_DateTime AS drop_off_time,
        Passenger_Count AS passenger_count,
        Trip_Distance AS trip_distance,
        Payment_Type AS payment_type,
        Total_Amt AS total_amount,
        Tip_Amt AS tip_amount,
        CASE Payment_Type
            WHEN  'Credit' THEN 1
            WHEN  'CREDIT' THEN 1
            ELSE 2
        END AS Payment_Category
    FROM 'C:/Users/ekadw/Documents/DATA/NY_Taxi/2009/yellow_taxi_2009/yellow_tripdata_*.parquet'
    WHERE Trip_Pickup_DateTime IS NOT NULL
        AND Trip_Dropoff_DateTime IS NOT NULL
        AND Passenger_Count >= 0
        AND Trip_Distance >= 0
        AND Trip_Distance <= 50
        AND Payment_Type IS NOT NULL
        AND Total_Amt >= 0
        AND Tip_Amt IS >= 0
        AND Trip_Pickup_DateTime >= '2009-01-01' 
        AND Trip_Pickup_DateTime < '2010-10-01'
), CTE_yellow_2010 AS (
    SELECT 
        pickup_datetime AS pick_up_time,
        dropoff_datetime AS drop_off_time,
        passenger_count,
        trip_distance,
        payment_type,
        total_amount,
        tip_amount
    FROM 'C:/Users/ekadw/Documents/DATA/NY_Taxi/2010/yellow_taxi_2010/yellow_tripdata_*.parquet'
    WHERE pickup_datetime IS NOT NULL
        AND dropoff_datetime IS NOT NULL
        AND passenger_count IS NOT NULL
        AND trip_distance IS NOT NULL
        AND payment_type IS NOT NULL
        AND total_amount IS NOT NULL
        AND tip_amount IS NOT NULL
        AND pickup_datetime >= '2010-01-01' 
        AND pickup_datetime < '2011-01-01'
), CTE_yellow_2011_2023 AS (
    SELECT 
        tpep_pickup_datetime AS pick_up_time,
        tpep_dropoff_datetime AS drop_off_time,
        passenger_count,
        trip_distance,
        payment_type,
        total_amount,
        tip_amount
    FROM 'C:/Users/ekadw/Documents/DATA/NY_Taxi/*/yellow_taxi/yellow_tripdata_*.parquet'
    WHERE tpep_pickup_datetime IS NOT NULL
        AND tpep_dropoff_datetime IS NOT NULL
        AND passenger_count IS NOT NULL
        AND trip_distance IS NOT NULL
        AND payment_type IS NOT NULL
        AND total_amount IS NOT NULL
        AND tip_amount IS NOT NULL
        AND tpep_pickup_datetime >= '2011-01-01' 
        AND tpep_dropoff_datetime < '2023-10-01'
), CTE_union_yellow_2009_2023 AS (
    SELECT * FROM CTE_yellow_2009
    UNION ALL
    SELECT * FROM CTE_yellow_2010
    UNION ALL
    SELECT * FROM CTE_yellow_2011_2023
), CTE_count_trip AS (
    SELECT
        DATE(pick_up_time) AS pick_up_date
    FROM CTE_union_yellow_2009_2023
)

SELECT 
    pick_up_date AS day_of_trip,
    COUNT(*) AS total_trip
FROM CTE_count_trip
GROUP BY pick_up_date
ORDER BY pick_up_date 
"""

con = duckdb.connect()
df_classification = con.execute(query_classification).fetchdf()
df_classification.head()